# Multi-Class Sentiment Classification
**Cameron Batts**

Python · NLP · Bag of Words · Naive Bayes · Logistic Regression · TextBlob

## Overview
Compare three sentiment classifiers on 40,000 social-media posts with 13 original emotion labels mapped to negative, neutral, and positive classes. The workflow includes preprocessing, training-only vocabulary fitting, an 80/20 split, and class-level evaluation.

Selected numerical outputs come from the completed original run. The cleaned workflow has been syntax-checked but not re-executed. The optional validation-based threshold search below is revised code and has no reported result.

## Data source
CrowdFlower sentiment analysis in text: https://data.world/crowdflower/sentiment-analysis-in-text . Source attribution is retained. Raw text and author identifiers are not included in this repository. Place a permitted local copy of `text_emotion.csv` in `data/` before running; only `sentiment` and `content` columns are needed. The dataset contains third-party names in its source text; no source-text examples are displayed here.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords as nltk_stopwords
from textblob import TextBlob
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn import naive_bayes
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import text

nltk.download('stopwords', quiet=True)
stopwords = nltk_stopwords.words('english')
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_path = project_root / 'data' / 'text_emotion.csv'
emotion_raw_csv = pd.read_csv(data_path, usecols=['sentiment', 'content'])
assert emotion_raw_csv['content'].notna().all(), 'Missing content must be reviewed.'
print('Rows:', len(emotion_raw_csv))


## Label mapping
Positive: enthusiasm, love, fun, happiness, relief. Negative: empty, sadness, worry, hate, boredom, anger. Neutral: neutral and surprise.

This mapping is an analytical choice. Surprise can carry either polarity, and treating empty as negative can introduce label ambiguity.

In [ ]:
emotions_dict = {
    'empty': -1,
    'sadness': -1,
    'enthusiasm': 1,
    'neutral': 0,
    'worry': -1,
    'surprise': 0,
    'love': 1,
    'fun': 1,
    'hate': -1,
    'happiness': 1,
    'boredom': -1,
    'relief': 1,
    'anger': -1
    }

In [ ]:
emotion_raw_csv['y'] = emotion_raw_csv['sentiment'].map(emotions_dict)
assert emotion_raw_csv['y'].notna().all(), 'Unexpected emotion labels.'
y = emotion_raw_csv['y']
print(y.value_counts().sort_index())


## Preprocessing and split
The baseline lowercases text, removes HTTP URLs, punctuation and numbers, and removes NLTK English stopwords. It does not explicitly remove complete @mentions: stripping punctuation can leave username tokens. Removing negation and emoticons may discard useful sentiment cues. These choices are retained to match the original experiment.

The original split uses `random_state=42` and a 20% test set, without stratification. The vectorizer is fit only on training text.

In [ ]:
def preprocess(text):
    # lowercase all text
    text = text.lower()
    # remove URLs
    text = re.sub(r'http\S+', '', text)
    # remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # split into words and remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stopwords]
    return ' '.join(tokens)

emotion_raw_csv['processed_content'] = emotion_raw_csv['content'].apply(preprocess)



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    emotion_raw_csv['processed_content'], y, test_size=0.2, random_state=42)


In [ ]:
vectorizer = text.CountVectorizer(stop_words=stopwords)


In [ ]:
# You fit and transform your training text and just transform the test text
# fit_transform builds the dictionary and transforms the text
# transform function uses the existing dictionary to transform text
# You must use the same dictionary or else the model shape will not match
X_train_vectors = vectorizer.fit_transform(X_train)
X_test_vectors = vectorizer.transform(X_test)

In [ ]:
print(len(vectorizer.get_feature_names_out()))


42633


## Naive Bayes
Evaluate per-class metrics in the order negative, neutral, positive.

In [ ]:
model = naive_bayes.MultinomialNB()
model.fit(X_train_vectors, y_train)
y_pred = model.predict(X_test_vectors)


In [ ]:
# Get our performance metrics, precision, recall, F1

precision_class = precision_score(y_test, y_pred, average=None, zero_division=0.0)
recall_class = recall_score(y_test, y_pred, average=None, zero_division=0.0)
f1_class = f1_score(y_test, y_pred, average=None, zero_division=0.0)

print("{:=^50s}".format("Naive-Bayes Performance by Class (entries are ordered as negative, neutral and positive)"))

# Output results by class
print("Precision:", precision_class)
print("Recall:", recall_class)
print("F1 Score:", f1_class)

Naive-Bayes Performance by Class (entries are ordered as negative, neutral and positive)
Precision: [0.54308317 0.44755245 0.58887677]
Recall: [0.79573935 0.1182448  0.61293984]
F1 Score: [0.64557123 0.18706613 0.60066741]


In [ ]:
precision_w = precision_score(y_test, y_pred, average='weighted', zero_division=0.0)
recall_w = recall_score(y_test, y_pred, average='weighted', zero_division=0.0)
f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0.0)

print("Weighted Precision:", precision_w)
print("Weighted Recall:", recall_w)
print("Weighted F1 Score:", f1_w)


Weighted Precision: 0.5323592307124395
Weighted Recall: 0.552
Weighted F1 Score: 0.5066531869624719


## Logistic regression
Use the same split and representation for a direct comparison.

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vectors, y_train)
y_pred_lr = lr_model.predict(X_test_vectors)


In [ ]:
precision_class = precision_score(y_test, y_pred_lr, average=None, zero_division=0.0)
recall_class = recall_score(y_test, y_pred_lr, average=None, zero_division=0.0)
f1_class = f1_score(y_test, y_pred_lr, average=None, zero_division=0.0)

print("Precision:", precision_class)
print("Recall:", recall_class)
print("F1 Score:", f1_class)


Precision: [0.60491329 0.42228886 0.60850729]
Recall: [0.65570175 0.39030023 0.584563  ]
F1 Score: [0.62928443 0.40566491 0.59629487]


In [ ]:
precision_w = precision_score(y_test, y_pred_lr, average='weighted', zero_division=0.0)
recall_w = recall_score(y_test, y_pred_lr, average='weighted', zero_division=0.0)
f1_w = f1_score(y_test, y_pred_lr, average='weighted', zero_division=0.0)

print("Weighted Precision:", precision_w)
print("Weighted Recall:", recall_w)
print("Weighted F1 Score:", f1_w)


Weighted Precision: 0.5566779208858396
Weighted Recall: 0.560375
Weighted F1 Score: 0.5578684677938621


## TextBlob baseline
Classify polarity below −0.33 as negative, above +0.33 as positive, and the remainder as neutral. Apply TextBlob to the same processed test text.

In [ ]:
y_pred_textblob_score = X_test.apply(lambda x: TextBlob(str(x)).sentiment[0])

In [ ]:
def classify_textblob(score):
    if score > 0.33:
        return 1
    elif score < -0.33:
        return -1
    else:
        return 0

y_pred_textblob = y_pred_textblob_score.apply(classify_textblob)


In [ ]:
precision_w = precision_score(y_test, y_pred_textblob, average='weighted', zero_division=0.0)
recall_w = recall_score(y_test, y_pred_textblob, average='weighted', zero_division=0.0)
f1_w = f1_score(y_test, y_pred_textblob, average='weighted', zero_division=0.0)

print("Weighted Precision:", precision_w)
print("Weighted Recall:", recall_w)
print("Weighted F1 Score:", f1_w)


Weighted Precision: 0.5780281173745957
Weighted Recall: 0.413625
Weighted F1 Score: 0.38943124217884945


## Test-set findings
Logistic regression has the highest weighted F1 among these three evaluated configurations: 0.5579 versus 0.5067 for Naive Bayes and 0.3894 for TextBlob. Its neutral-class F1 is 0.4057 compared with 0.1871 for Naive Bayes. Naive Bayes has slightly higher negative-class F1 (0.6456 versus 0.6293).

The result supports logistic regression for this experiment, not a universal claim about imbalanced text datasets. Accuracy remains modest, and the single random split provides no uncertainty interval.

## Optional extension: validation-based TextBlob thresholds
The original optional search evaluated thresholds on training labels and coupled the positive cutoff with the magnitude of the negative cutoff. This revision independently searches negative and positive thresholds using a held-out portion of the training set. The test set is used only after selecting thresholds. This extension has not been executed; its results must not be confused with the baseline test metrics above.

In [ ]:
from itertools import product
threshold_train, threshold_valid, labels_train, labels_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
valid_scores = threshold_valid.apply(lambda value: TextBlob(str(value)).sentiment.polarity).to_numpy()
best_score = -1.0
best_thresholds = None
for negative_cutoff, positive_cutoff in product(np.arange(-0.55, 0, 0.05), np.arange(0.05, 0.60, 0.05)):
    predictions = np.where(valid_scores > positive_cutoff, 1,
                           np.where(valid_scores < negative_cutoff, -1, 0))
    score = f1_score(labels_valid, predictions, average='weighted', zero_division=0)
    if score > best_score:
        best_score = score
        best_thresholds = (negative_cutoff, positive_cutoff)
negative_cutoff, positive_cutoff = best_thresholds
test_scores = y_pred_textblob_score.to_numpy()
tuned_predictions = np.where(test_scores > positive_cutoff, 1,
                             np.where(test_scores < negative_cutoff, -1, 0))
print('Validation-selected thresholds:', best_thresholds)
print('Validation weighted F1:', best_score)
print('Tuned test weighted F1:', f1_score(y_test, tuned_predictions, average='weighted', zero_division=0))


## Limitations and next steps
- Emotion-to-sentiment mapping is subjective, particularly for surprise and empty.
- A random row split does not control for duplicate text or repeated authors across train and test.
- Preprocessing can remove negation and emoticons and retain mention text.
- Compare stratified and grouped splits, negation-preserving preprocessing, n-grams, TF-IDF, and class weighting using validation data.
- These results describe social-media text; performance on healthcare feedback has not been measured.

## Author
Cameron Batts